# Tutorial: TriShift on a custom AnnData

This notebook shows the smallest end-to-end TriShift workflow on a user-provided `AnnData` object. It uses a synthetic toy dataset so that the full pipeline remains easy to inspect and fast to run.

You will see five steps:
1. build a custom `AnnData` and a matching gene embedding table,
2. initialize `TriShiftData`,
3. initialize and train a small `TriShift` model,
4. evaluate on a held-out condition split,
5. export prediction payloads for downstream analysis.


## Prerequisites

- Run `pip install -e .` from the repository root.
- Use a Python environment with `torch`, `anndata`, `scanpy`, `numpy`, and `pandas` available.
- The notebook uses CPU by default so it can run on a laptop-sized example.


In [ ]:
from __future__ import annotations

from pathlib import Path
import tempfile

import anndata as ad
import numpy as np
import pandas as pd
import torch

from trishift import TriShift, TriShiftData

SEED = 21
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
SEED


## Step 1. Build a tiny custom `AnnData`

TriShift expects a cell-by-gene matrix and a condition column. In this toy example we create one control condition and three perturbation conditions (`A`, `B`, `A+B`).


In [ ]:
n_ctrl = 20
n_per = 12
n_genes = 40

X_ctrl = rng.random((n_ctrl, n_genes), dtype=np.float32)
X_a = rng.random((n_per, n_genes), dtype=np.float32)
X_b = rng.random((n_per, n_genes), dtype=np.float32)
X_ab = rng.random((n_per, n_genes), dtype=np.float32)
X = np.vstack([X_ctrl, X_a, X_b, X_ab])

obs = pd.DataFrame({
    "condition": ["ctrl"] * n_ctrl + ["A"] * n_per + ["B"] * n_per + ["A+B"] * n_per
})
var = pd.DataFrame({"gene_name": ["A", "B"] + [f"G{i}" for i in range(n_genes - 2)]})
adata = ad.AnnData(X=X, obs=obs, var=var)

# Minimal DEG metadata keeps the tiny example self-contained.
top20 = np.arange(min(20, n_genes), dtype=int)
adata.uns["top20_degs_non_dropout"] = {
    "A": top20.copy(),
    "B": top20.copy(),
    "A+B": top20.copy(),
}
adata.uns["gene_idx_non_zeros"] = {
    "A": top20.copy(),
    "B": top20.copy(),
    "A+B": top20.copy(),
}
adata.uns["gene_idx_non_dropout"] = {
    "A": top20.copy(),
    "B": top20.copy(),
    "A+B": top20.copy(),
}

# The embedding table index should contain ctrl and all genes used in condition tokens.
emb_dim = 8
embd_df = pd.DataFrame(
    rng.random((3, emb_dim), dtype=np.float32),
    index=["ctrl", "A", "B"],
)

adata


## Step 2. Wrap the data with `TriShiftData`

`TriShiftData` aligns conditions, embeddings, DEG caches, and split metadata.


In [ ]:
data = TriShiftData(adata=adata, embd_df=embd_df, label_key="condition", var_gene_key="gene_name")
data.setup_embedding_index()
data.build_or_load_degs()

split = data.split_by_condition(seed=SEED, test_ratio=0.2, val_ratio=0.2)
{k: getattr(v, "shape", len(v) if v is not None else None) for k, v in split.items() if k in ["train", "val", "test", "train_conds", "val_conds", "test_conds"]}


## Step 3. Initialize and train a tiny TriShift model

The hidden dimensions and epoch counts below are intentionally small. They are suitable for a smoke test, not for benchmark-quality runs.


In [ ]:
model = TriShift(data, device="cpu")
model.model_init(
    x_dim=adata.n_vars,
    z_dim=8,
    cond_dim=embd_df.shape[1],
    vae_enc_hidden=[16],
    vae_dec_hidden=[16],
    shift_hidden=[16],
    gen_hidden=[16],
    dropout=0.0,
)

stage1_logs = model.train_stage1_vae(
    data.adata_ctrl,
    epochs=1,
    batch_size=8,
    lr=1e-3,
    amp=False,
    num_workers=0,
    pin_memory=False,
    grad_accum_steps=1,
)
model.encode_and_cache_mu(
    data.adata_all,
    batch_size=16,
    amp=False,
    num_workers=0,
    pin_memory=False,
)
stage1_logs["epochs"][-1]


## Step 4. Train the shift/generator stage and evaluate held-out conditions

For a custom dataset, this is the first point where you can inspect condition-level metrics such as Pearson and nMSE.


In [ ]:
emb_table = torch.tensor(data.embd_df.values, dtype=torch.float32)

stage23_logs = model.train_stage23_joint(
    split_dict=split,
    emb_table=emb_table,
    mode="knn",
    k=3,
    split_id=1,
    epochs=1,
    batch_size=4,
    lr=1e-3,
    amp=False,
    num_workers=0,
    pin_memory=False,
    grad_accum_steps=1,
)

metrics_df = model.evaluate(
    split_dict=split,
    emb_table=emb_table,
    split_id=1,
    n_ensemble=10,
    base_seed=24,
)

metrics_df[["condition", "pearson", "nmse", "deg_mean_r2"]]


## Step 5. Export prediction payloads

The exported dictionary contains the arrays used later by downstream figures and evaluation scripts.


In [ ]:
tmp_dir = Path(tempfile.mkdtemp(prefix="trishift_tutorial_"))
out_path = tmp_dir / "tutorial_predictions.pkl"
preds = model.export_predictions(
    split_dict=split,
    emb_table=emb_table,
    split_id=1,
    n_ensemble=10,
    base_seed=24,
    out_path=str(out_path),
)
print(out_path)
list(preds.keys())[:3]


## Where to go next

- Replace the toy `AnnData` with your own preprocessed single-cell perturbation dataset.
- Replace the tiny embedding table with the gene embedding resource used in your experiment.
- Increase the hidden dimensions, `k`, and epoch counts before running a real benchmark.
- Use the dataset-specific scripts under `scripts/trishift/<dataset>/` as templates for full-scale training runs.
